# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [6]:
col = "Changed_Credit_Limit"

## <font color = 'skyblue'> ANÁLISIS GENERAL

Las medianas tienen el orden esperado: Median Bad > Mediana Standard > Mediana Good.

Esto concuerda con la hipótesis de que a mayor el cambio porcentual en el límite de crédito mayor cantidad de malos deudores.

In [8]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [9]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [10]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Changed_Credit_Limit_Decile,,,,,,,,,,
0,-6.49,2.28,10045,0.10045,2214,5405,2426,0.220408,0.538079,0.241513
1,2.29,4.33,9964,0.09964,2171,5247,2546,0.217884,0.526596,0.255520
2,4.34,6.34,9994,0.09994,2200,5094,2700,0.220132,0.509706,0.270162
3,6.35,7.98,10030,0.10030,1837,4038,4155,0.183151,0.402592,0.414257
4,7.99,9.40,10000,0.10000,1723,3542,4735,0.172300,0.354200,0.473500
5,9.41,10.99,10012,0.10012,878,3684,5450,0.087695,0.367958,0.544347
6,11.00,13.10,9962,0.09962,553,2696,6713,0.055511,0.270628,0.673861
7,13.11,16.37,10004,0.10004,1617,505,7882,0.161635,0.050480,0.787885
8,16.38,19.44,10018,0.10018,2723,173,7122,0.271811,0.017269,0.710920


No missing values found.
No infinite values found.
No duplicate rows found.


In [14]:
df[(df[continuous_variable] >= 19.45) & (df[continuous_variable] <= 36.97) & (df['Credit_Score'] == 1)].shape

(2119, 85)

Proporción de Buenos: se observa una relación negativa entre los cambios porcentuales en el límite de crédito y la proporción de buenos deudores, lo cual es razonable.

Proporción de standard: esta relación no es tan clara hasta cambios porcentuales por encima del 14%, pero luego de este umbral, sí, lo cual es razonable.

Proporción de malos: la proporción de malos y los cambios porcentuales en el límite del crédito no es la esperanda para cambios inferiores al 13%, aproximadamente, luego de este umbral sí, lo cual es razonable.

In [11]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y la variable:

In [16]:
group_map = {0: "Group_1", 
             1: "Group_1", 
             2: "Group_1", 
             3: "Group_1", 
             4: "Group_1",
             5: "Group_1", 
             6: "Group_1",
             7: "Group_1",
             8: "Group_2",
             9: "Group_3"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Changed_Credit_Limit,,,,,,,,,,
Group_1,-6.49,16.37,80011,0.80011,13193,30211,36607,0.164890,0.377586,0.457525
Group_2,16.38,19.44,10018,0.10018,2723,173,7122,0.271811,0.017269,0.710920
Group_3,19.45,36.97,9971,0.09971,7852,0,2119,0.787484,0.000000,0.212516


No missing values found.
No infinite values found.
No duplicate rows found.


In [18]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [19]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [20]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Changed_Credit_Limit,"100,000.00",10.39,6.78,-6.49,5.34,9.40,14.85,36.97


Todos los coeficientes son significativos.

Changed_Credit_Limit_Scaled -6.2643: según lo esperado, el coeficiente es negativo: por cada punto porcentual de aumento en los límites de crédito, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -3.8657: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.8618: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [21]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.951491
         Iterations: 13
         Function evaluations: 15
         Gradient evaluations: 15
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -95149.
Model:                   OrderedModel   AIC:                         1.903e+05
Method:            Maximum Likelihood   BIC:                         1.903e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        11:59:16                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                  coef   

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Changed_Credit_Limit_Decile -0.2833: Por cada punto porcencual de aumento en los límties de crédito, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -2.6621: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.8221: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [23]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.979961
         Iterations: 11
         Function evaluations: 13
         Gradient evaluations: 13
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -97996.
Model:                   OrderedModel   AIC:                         1.960e+05
Method:            Maximum Likelihood   BIC:                         1.960e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        12:01:56                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                  coef   

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Changed_Credit_Limit -1.4635: Por cada grupo adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -3.2501: Umbral que separa las categorías Bad y Standard.

Threshold 1/2 0.8466: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [24]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.957463
         Iterations: 11
         Function evaluations: 12
         Gradient evaluations: 12
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -95746.
Model:                   OrderedModel   AIC:                         1.915e+05
Method:            Maximum Likelihood   BIC:                         1.915e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        12:02:07                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                   coef  

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Changed_Credit_Limit` usando Regresión Ordinal

| Representación                          | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|----------------------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Changed_Credit_Limit_Scaled`          | -6.2643               | **Log-Likelihood**: -95,149<br>**AIC**: 190,298<br>**BIC**: 190,339                    | 🔹 Mejor ajuste.<br>🔹 Relación negativa clara con el score.<br>🔹 Conserva variabilidad completa. |
| `Changed_Credit_Limit_Decile`          | -0.2833               | **Log-Likelihood**: -97,996<br>**AIC**: 195,992<br>**BIC**: 196,033                    | 🔹 Peor ajuste.<br>🔹 Discretización en deciles reduce precisión explicativa. |
| `Grouped_Changed_Credit_Limit`         | -1.4635               | **Log-Likelihood**: -95,746<br>**AIC**: 191,492<br>**BIC**: 191,533                    | 🔹 Ajuste intermedio.<br>🔹 Balance entre interpretabilidad y desempeño. |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [42]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
